# Complementarity-aware sample selection on NYU Depth V2

This notebook picks which 5% / 10% of the training images to label with our **label-free depth-edge
complementarity score C** and with the baselines, then trains the same model as the random baseline on each pick.

**Needs:** the baseline notebook already finished (it is: `MyDrive/rgbx/runs/` holds the 14 runs and their weights),
and the updated `rgbx-baseline` folder uploaded to `MyDrive/rgbx/` (replace the old one).
Runtime → Change runtime type → **T4 GPU**.

| step | what | time on a T4 |
|---|---|---|
| 1 to 3 | connect Drive, get code, unzip data | ~3 min |
| 4 | compute all scores (no training) | ~5 min |
| 5 | sanity check: does C point at images where depth helps? | ~5 min |
| 6 | look at the images C likes and dislikes | 1 min |
| 7 | write the split files | 1 min |
| 8 | **priority 1 runs**: 18 runs (6 cold start at 5%, 12 round 2 at 10%) | ~30 min each, ~9 h total |
| 9 | priority 2 runs (optional ablations) | ~30 min each |
| 10 | results table | 1 min |

Every finished run is skipped next time, so spread step 8 over several sessions: each time run cells 1 to 3, then step 8.
Steps 4 and 7 only need to run once (their outputs are kept on Drive).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rgbx'
import os; os.makedirs(DRIVE, exist_ok=True)

## 2. Get the code
Upload the updated `rgbx-baseline` folder (unzipped) into `MyDrive/rgbx/` first, or put a GitHub URL below.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<your-username>/rgbx-segmentation.git'
CODE = '/content/rgbx-baseline'
if REPO_URL:
    !rm -rf {CODE} && git clone -q {REPO_URL} {CODE}
else:
    !rm -rf {CODE} && cp -r {DRIVE}/rgbx-baseline {CODE}
%cd {CODE}
!pip -q install timm
!ls
!pip -q install scikit-learn

## 3. Unzip the prepared data (made by the baseline notebook)

In [ ]:
import os
DATA = '/content/data/nyuv2'
ZIP = f'{DRIVE}/nyuv2_prepared.zip'
if os.path.exists(ZIP):
    !mkdir -p /content/data && unzip -q -o {ZIP} -d /content/data
else:
    MAT = f'{DRIVE}/nyu_depth_v2_labeled.mat'
    if not os.path.exists(MAT):
        MAT = '/content/nyu_depth_v2_labeled.mat'
        !wget -c -q --show-progress -O {MAT} http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
    !python prepare_nyuv2.py --mat {MAT} --out {DATA}
    !cd /content/data && zip -q -r {ZIP} nyuv2
!ls {DATA}/train/rgb | wc -l; ls {DATA}/test/rgb | wc -l   # expect 795 and 654
RUNS, CACHE, SPLITS = f'{DRIVE}/runs', f'{DRIVE}/cache', f'{DRIVE}/splits_sel'

## 4. Compute the scores (once, ~5 minutes, no training)
* `labelfree_train.npz`: label-free C for all 795 training images, plus ImageNet features for diversity.
* `model_r5_s{0,1,2}.npz`: entropy, model-based C (RGB vs RGB-D disagreement) and features, from the
  baseline models already trained on the random 5% (`rand5_rgbd_s*` and `rand5_rgb_s*`). Used only for round 2.

In [ ]:
!python complementarity.py --data {DATA} --split train --out {CACHE}/labelfree_train.npz
for s in (0, 1, 2):
    !python model_scores.py --data {DATA} --runs {RUNS} --seed {s} --out {CACHE}/model_r5_s{s}.npz

## 5. Sanity check before spending GPU hours (~5 minutes)
On the 654 test images, compares the two full-data models you already trained (RGB-D vs RGB-only) image by image,
and checks whether images with a high C are the ones where depth helps most.
**Please paste the printed lines back into the project thread.** A positive Spearman for `C_mean` that is higher than
for `depth_edge` is good news; a value near zero means we should rethink C before step 8.

In [ ]:
!python check_c.py --data {DATA} --runs {RUNS} --cache {CACHE}

## 6. Look at what C likes (top row) and dislikes (bottom row)
The right-hand map shows depth edges that RGB does not show (bright = high complementarity).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from complementarity import edge_maps, complementarity_map, load_pair
z = np.load(f'{CACHE}/labelfree_train.npz')
order = np.argsort(-z['C_mean'])
fig, ax = plt.subplots(4, 3, figsize=(13, 13))
for r, i in enumerate([order[0], order[1], order[-2], order[-1]]):
    rgb, depth = load_pair(f'{DATA}/train', int(i))
    m, e_d, e_r = complementarity_map(*edge_maps(rgb, depth), float(z['p_d']), float(z['p_r']))
    ax[r, 0].imshow(rgb); ax[r, 0].set_title(f'train #{i}  C = {z["C_mean"][i]:.4f}')
    ax[r, 1].imshow(depth / 1000, cmap='viridis'); ax[r, 1].set_title('depth')
    ax[r, 2].imshow(m, cmap='magma', vmin=0, vmax=1); ax[r, 2].set_title('depth edge missing in RGB')
[a.axis('off') for a in ax.ravel()]; plt.tight_layout(); plt.show()
plt.hist(z['C_mean'], 50); plt.xlabel('C_mean'); plt.ylabel('images'); plt.show()

## 7. Write the split files (once)
Cold start, 5%: `ctop` (C only), `kmeans` (diversity only), `cdiv` (ours, C + diversity).
Round 2, 5% → 10%: keep the random 5% of the seed and add 40 images by `entropy`, `coreset`, `mcomp` (model-based C),
`mcdiv`, `cdiv` (ours, label-free), plus the label-free ablations `ctop` and `kmeans`.
The files go to `MyDrive/rgbx/splits_sel/`. **Please also download that folder and upload it to the project**, so the
exact picks are on record.

In [ ]:
!python select_subsets.py --cache {CACHE} --out-dir {SPLITS} --methods ctop kmeans cdiv --frac 0.05
!python select_subsets.py --cache {CACHE} --out-dir {SPLITS} --round2 --from-dir splits --methods entropy coreset mcomp mcdiv cdiv ctop kmeans --frac 0.10
!ls {SPLITS} | wc -l   # expect 30

## 8. Priority 1 runs (RGB-D, 18 runs, ~9 h in total)
Order: the cold-start comparison first (our C vs diversity only), then round 2 against the standard baselines.
Random 5% and random 10% are already done in the baseline, with the same seeds and settings.

In [ ]:
!python run_selection.py --data {DATA} --out {RUNS} --splits-dir {SPLITS} --jobs cdiv:0.05 kmeans:0.05 r2-cdiv:0.10 r2-entropy:0.10 r2-coreset:0.10 r2-mcomp:0.10

## 9. Priority 2 runs (optional ablations, only after step 8)
* `ctop` at 5% and `r2-ctop`: C without diversity (shows why the clustering matters).
* `r2-mcdiv`: model-based C with diversity. `r2-kmeans`: diversity only in round 2.
* The second cell trains the RGB-only model on our 5% picks (does C help RGB-D more than RGB?).

In [ ]:
!python run_selection.py --data {DATA} --out {RUNS} --splits-dir {SPLITS} --jobs ctop:0.05 r2-mcdiv:0.10 r2-ctop:0.10 r2-kmeans:0.10

In [ ]:
!python run_selection.py --data {DATA} --out {RUNS} --splits-dir {SPLITS} --jobs cdiv:0.05 kmeans:0.05 --modes rgb

## 10. Results table
Paste this (or upload `MyDrive/rgbx/runs/summary.md` and `results.csv`) into the project thread.

In [ ]:
!python summarize.py --csv {RUNS}/results.csv --md {RUNS}/summary.md